In [439]:
import pandas as pd
import unicodedata
import re
import numpy as np

# 0 Funcoes

## Funções Bryan

In [440]:
# Como estamos importando de um excel, vamos ter que ajustar o nome das colunas
def normalizar_colunas(df):
    def remover_acentos(texto):
        return ''.join(
            c for c in unicodedata.normalize('NFKD', texto)
            if not unicodedata.combining(c)
        )

    df = df.copy()
    df.columns = [
        remover_acentos(col)
            .lower()
            .replace(' ', '_')
        for col in df.columns
    ]

    return df


In [441]:
# Como vamos juntar alguns dataframes, é melhor que estejam com os mesmos nomes algumas colunas
def renomear_colunas(df, mapa_colunas):
    """
    Parâmetros:
    df (pd.DataFrame): DataFrame original
    mapa_colunas (dict): {'nome_antigo': 'nome_novo'}

    Retorna:
    pd.DataFrame: DataFrame com colunas renomeadas
    """
    df = df.copy()

    # Aplica somente às colunas que existem no DataFrame
    mapa_valido = {
        col_antiga: col_nova
        for col_antiga, col_nova in mapa_colunas.items()
        if col_antiga in df.columns
    }

    df.rename(columns=mapa_valido, inplace=True)
    return df


In [442]:
"""
Como os dataframes tem colunas diferentes, na hora de juntar essa função irá ajudar.
Iremos adicionar colunas nos outros dataframes para que a junção possa ocorrer (as colunas terão dasdos vazios)
"""
def adaptar_dataframe(df, colunas_base, origem, lista_ids):
    df=df.copy()

     # filtra apenas os IDs desejados
    df = df[df['ra'].isin(lista_ids)]

    # adiciona colunas que faltam
    for col in colunas_base:
        if col not in df.columns:
            df[col] = pd.NA

    # mantém apenas as colunas do principal
    df = df[colunas_base]

    # cria coluna de origem
    df['ano_dataframe'] = origem

    return df


In [443]:
# Função correção de tipo de colunas
def corrigir_dados(tipo, dataframe, colunas):
    """
    Corrige o tipo de dados de colunas de um DataFrame.

    Parâmetros:
    tipo (str): tipo alvo ('int', 'float', 'str', 'data')
    dataframe (pd.DataFrame): DataFrame original
    colunas (list ou str): coluna ou lista de colunas

    Retorna:
    pd.DataFrame: DataFrame com colunas corrigidas
    """
    df = dataframe.copy()

    if isinstance(colunas, str):
        colunas = [colunas]

    match tipo.lower():
        case 'int':
            for col in colunas:
                df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

        case 'float':
            for col in colunas:
                df[col] = pd.to_numeric(df[col], errors='coerce')

        case 'str':
            for col in colunas:
                df[col] = df[col].astype(str).str.strip()

        case 'data' | 'datetime':
            for col in colunas:
                df[col] = pd.to_datetime(df[col], errors='coerce')

        case _:
            raise ValueError(
                f"Tipo '{tipo}' não suportado. "
                "Use: int, float, str, data"
            )

    return df

In [444]:
def colunas_totalmente_vazias(df):
    """
    Identifica colunas que possuem apenas valores vazios e deleta

    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada
    remover (bool): se True, remove as colunas vazias

    Retorna:
    pd.DataFrame:
        - DataFrame sem colunas vazias (remover=True)
    """
    df_tmp = df.copy()

    # considera strings vazias como NaN
    df_tmp = df_tmp.replace(r'^\s*$', pd.NA, regex=True)

    colunas_vazias = [
        col for col in df_tmp.columns
        if df_tmp[col].isna().all()
    ]

    return df_tmp.drop(columns=colunas_vazias)


In [445]:
def arredondar_floats(df, casas=2):
    """
    Arredonda todas as colunas float de um DataFrame.

    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada
    casas (int): número de casas decimais (padrão = 2)

    Retorna:
    pd.DataFrame: DataFrame com floats arredondados
    """
    df = df.copy()

    colunas_float = df.select_dtypes(include=['float', 'float64', 'float32']).columns

    df[colunas_float] = df[colunas_float].round(casas)

    return df

## Funções Vitor

In [446]:

def normalizar_fase(valor):
    """
    Converte valores como '1A', '2C', '8F' ou 9 em 'FASE X'.
    Mantém valores sem número (ex: 'ALFA').
    """
    valor_str = str(valor)
    numeros = "".join(filter(str.isdigit, valor_str))
    
    if numeros:
        return f"FASE {numeros}"
    else:
        return valor

In [447]:
def remover_texto_parenteses(valor):
    """
    Remove qualquer texto entre parênteses e retorna o texto em MAIÚSCULO.
    Ex: 'Fase 1 (3° e 4° ano)' , 'Fase 2 (5° e 6° ano)', 'Fase 3 (7° e 8° ano)', 'Fase 4 (9° ano)',
       'Fase 6 (2° EM)', 'Fase 5 (1° EM)', 'Fase 7 (3° EM)',
       'Fase 8 (Universitários)' -> 'FASE 1', 'FASE 2', 'FASE 3', etc
    """
    if pd.isna(valor):
        return valor
    
    texto_limpo = re.sub(r"\s*\(.*?\)", "", str(valor)).strip()
    return texto_limpo.upper()

## Funções Luis

In [448]:
def genero_norm(v):
    if pd.isna(v): return np.nan
    s = str(v).strip().lower()
    if s in ["menina","feminino","f"]: return "Feminino"
    if s in ["menino","masculino","m"]: return "Masculino"
    return str(v).strip()

# 1 Importando os dados

In [449]:
arquivo = r"https://raw.githubusercontent.com/vbomura/tc5/0afbe2d22ee4658c81a7470b2360a2aef3cd032e/Base_Passos_Magicos/BASE%20DE%20DADOS%20PEDE%202024%20-%20DATATHON.xlsx"

# Pegando o dado de cada aba
base_2022 = pd.read_excel(arquivo, sheet_name="PEDE2022")
base_2023 = pd.read_excel(arquivo, sheet_name="PEDE2023")
base_2024 = pd.read_excel(arquivo, sheet_name="PEDE2024")

In [450]:
base_2022.columns

Index(['RA', 'Fase', 'Turma', 'Nome', 'Ano nasc', 'Idade 22', 'Gênero',
       'Ano ingresso', 'Instituição de ensino', 'Pedra 20', 'Pedra 21',
       'Pedra 22', 'INDE 22', 'Cg', 'Cf', 'Ct', 'Nº Av', 'Avaliador1',
       'Rec Av1', 'Avaliador2', 'Rec Av2', 'Avaliador3', 'Rec Av3',
       'Avaliador4', 'Rec Av4', 'IAA', 'IEG', 'IPS', 'Rec Psicologia', 'IDA',
       'Matem', 'Portug', 'Inglês', 'Indicado', 'Atingiu PV', 'IPV', 'IAN',
       'Fase ideal', 'Defas', 'Destaque IEG', 'Destaque IDA', 'Destaque IPV'],
      dtype='object')

In [451]:
#Ajustando nomes identicos entre as baeses
base_2022 = base_2022.rename(columns={
    'Nome': 'Nome Anonimizado',
    'Idade 22': 'Idade',
    'Matem': 'Mat',
    'Portug': 'Por',
    'Inglês': 'Ing',
    'Defas': 'Defasagem',
    'Pedra 22': 'Pedra',
    'INDE 22': 'INDE'
})


#Ajustando nomes identicos entre as baeses
base_2023 = base_2023.rename(columns={
    'Pedra 2023': 'Pedra',
    'INDE 2023': 'INDE',
    'Data de Nasc': 'Ano nasc'
})

#Ajustando nomes identicos entre as baeses
base_2024 = base_2024.rename(columns={
    'Pedra 2024': 'Pedra',
    'INDE 2024': 'INDE',
    'Data de Nasc': 'Ano nasc'
})

In [452]:
#Criando coluna para ANO
base_2022['Ano_Aba'] = 2022
base_2022['Ano_Aba'] = base_2022['Ano_Aba'].astype(int)

base_2023['Ano_Aba'] = 2023
base_2023['Ano_Aba'] = base_2023['Ano_Aba'].astype(int)

base_2024['Ano_Aba'] = 2024
base_2024['Ano_Aba'] = base_2024['Ano_Aba'].astype(int)

In [453]:
#remover colunas dos dataframes 2022 
#'Data de Nasc',
colunas_remover2022 = ['Cg','Cf','Ct','Pedra 20','Pedra 21'
                   ,'Avaliador1','Rec Av1','Avaliador2','Rec Av2','Avaliador3','Rec Av3','Avaliador4'
                   ,'Rec Av4','Rec Psicologia','Indicado','Atingiu PV','Destaque IEG','Destaque IDA'
                   ,'Destaque IPV']

base_2022.drop(columns=colunas_remover2022, inplace=True)

In [454]:
#remover colunas dos dataframes 2023 
#'Data de Nasc',
colunas_remover2023 = ['Pedra 20', 'Pedra 21','Pedra 22','Pedra 23','INDE 22','INDE 23','Cg','Cf','Ct'
                   ,'Avaliador1','Rec Av1','Avaliador2','Rec Av2','Avaliador3','Rec Av3','Avaliador4'
                   ,'Rec Av4','Rec Psicologia','Indicado','Atingiu PV','Destaque IEG','Destaque IDA'
                   ,'Destaque IPV','Destaque IPV.1']

base_2023.drop(columns=colunas_remover2023, inplace=True)

In [455]:
#remover colunas dos dataframes 2024 
#'Data de Nasc',
colunas_remover2024 = ['Pedra 20', 'Pedra 21','Pedra 22','Pedra 23','INDE 22','INDE 23','Cg','Cf','Ct'
                   ,'Avaliador1','Rec Av1','Avaliador2','Rec Av2','Avaliador3','Avaliador4','Avaliador5'
                   ,'Avaliador6', 'Rec Psicologia','Indicado','Atingiu PV','Destaque IEG','Destaque IDA'
                   ,'Destaque IPV','Escola','Ativo/ Inativo','Ativo/ Inativo.1']

base_2024.drop(columns=colunas_remover2024, inplace=True)

## Ajustes nas colunas

In [456]:
# Vamos padronizar os nomes das colunas
base_2022 = normalizar_colunas(base_2022)
base_2023 = normalizar_colunas(base_2023)
base_2024 = normalizar_colunas(base_2024)

### Tratamento validações para ajuste para identificar a idade dos alunos (ano de 2023 a coluna idade não está confiavel)

In [457]:
base_2022['ano_nascimento'] = base_2022['ano_nasc']

In [458]:
base_2022['ano_nascimento'].unique()

array([2003, 2005, 2004, 2002, 2001, 2006, 2007, 2009, 2008, 2010, 2011,
       2012, 2013, 2014, 2015])

In [459]:
base_2023['ano_nasc_novo'] = pd.to_datetime(
    base_2023['ano_nasc'], 
    errors='coerce'
)

base_2023['ano_nascimento'] = base_2023['ano_nasc_novo'].dt.year

In [460]:
base_2023['ano_nascimento'].unique()

array([2015, 2014, 2016, 2013, 2012, 2011, 2009, 2010, 2008, 2007, 2006,
       2005, 2004, 2001, 2003, 2002, 1996, 1999, 1998], dtype=int32)

In [461]:
base_2024['ano_nasc_novo'] = pd.to_datetime(
    base_2024['ano_nasc'], 
    errors='coerce'
)

base_2024['ano_nascimento'] = base_2024['ano_nasc_novo'].dt.year

In [462]:
base_2024['ano_nascimento'].unique()

array([2016, 2015, 2014, 2017, 2013, 2012, 2011, 2010, 2009, 2008, 2007,
       2006, 2005, 2002, 2003, 1996, 2004, 1999, 2001, 1998, 2000],
      dtype=int32)

### Tratamento para coluna FASE

<div class="alert alert-block alert-info">
⚠️ **IMPORTANTE VALIDAR  ALTERACAO QUE VITOR REALIZOU**
</div>

In [463]:
#Não identifiquei mas por "dedução fiz um de para na coluna FASE na base 2024"
""" 
Alterado de:
array(['ALFA', '1A', '1B', '1C', '1D', '1E', '1G', '1H', '1J', '1K', '1L', '1M', '1N', '1P', '1R', '2A', '2B', '2C', '2D', '2G', '2H'
        , '2I', '2K', '2L', '2M', '2N', '2P', '2R', '2U', '3A', '3B', '3C', '3D', '3F', '3G', '3H', '3I', '3K', '3L', '3M', '3N', '3P'
        , '3R', '3U', '4A', '4B', '4C', '4F', '4H', '4L', '4M', '4N', '4R', '5A', '5B', '5C', '5D', '5F', '5G', '5L', '5M', '5N', '6A'
        , '6L', '7A', '7E', '8A', '8B', '8D', '8E', '8F', 9], dtype=object) 

Para: array(['ALFA', 'FASE 1', 'FASE 2', 'FASE 3', 'FASE 4', 'FASE 5', 'FASE 6', 'FASE 7', 'FASE 8', 'FASE 9'], dtype=object) 
"""

base_2024 = base_2024.rename(columns={"fase": "fase_original"})
base_2024["fase"] = base_2024["fase_original"].map(normalizar_fase)

In [464]:
base_2024['fase'].value_counts().sort_index()


fase
ALFA      196
FASE 1    185
FASE 2    185
FASE 3    211
FASE 4    115
FASE 5    100
FASE 6     25
FASE 7     37
FASE 8     64
FASE 9     38
Name: count, dtype: int64

![Quantidade total alunos fases](AlunosFases_2024.jpg)


Fonte da imagem anterior com totais de alunos por fase de 2024, retirado do documento:

https://passosmagicos.org.br/wp-content/uploads/2025/05/relatorio_de_atividades_2024_compressed.pdf


### Temos uma diferença de 1 registro para a fase 4 (nosso de para chegou em 115 e no documento informa 114)

### Temos uma diferença de 1 registro para a fase 6 (nosso de para chegou em 25 e no documento informa 24)

<div class="alert alert-block alert-info">
⚠️ **PODEMOS TOMAR COMO CORRETO ESTE DE PARA FEITO???**
</div>

In [465]:
#Guardando informações origiais de fase_ideal antes de fazer o map

base_2022 = base_2022.rename(columns={"fase_ideal": "fase_ideal_original"})
base_2022["fase_ideal"] = base_2022["fase_ideal_original"].map(remover_texto_parenteses)

base_2023 = base_2023.rename(columns={"fase_ideal": "fase_ideal_original"})
base_2023["fase_ideal"] = base_2023["fase_ideal_original"].map(remover_texto_parenteses)

base_2024 = base_2024.rename(columns={"fase_ideal": "fase_ideal_original"})
base_2024["fase_ideal"] = base_2024["fase_ideal_original"].map(remover_texto_parenteses)

## Unificando as bases de 2022, 2023 e 2024

In [466]:
#Unificando todas as bases em um unico dataframe

base_anos = pd.concat(
    [base_2022, base_2023, base_2024],
    axis=0,        # empilha linhas
    ignore_index=True,
    sort=False     # mantém todas as colunas
)

### Ajustando coluna Idade

In [467]:
#Ajustando coluna de Ano nascimento para conseguir criar uma nova coluna de idade, pois na base de 2023 possuem dados que não são inteiros
colunas_base_anos = ['ano_nasc_novo']
base_anos.drop(columns=colunas_base_anos, inplace=True)

#guardando colunas originais
base_anos = base_anos.rename(columns={"ano_nasc": "ano_nasc_original"})
base_anos = base_anos.rename(columns={"idade": "idade_original"})

#Fazendo uma nova conta para pegar idade para aqueles onde não tinha informação de idade na coluna idade_original
base_anos['idade_new'] = base_anos['ano_aba']-base_anos['ano_nascimento']
base_anos = base_anos.rename(columns={"idade_new": "idade"})


### 

### Ajustando dados de Pedra (pois continha Ágata e Agata) além de padronizar a coluna genero

In [468]:
base_anos['pedra'].unique()

array(['Quartzo', 'Ametista', 'Ágata', 'Topázio', 'Agata', nan, 'INCLUIR'],
      dtype=object)

In [469]:
#Ajustando o nome da Pedra para mantermos um padrão definido no documento da Passos Magicos
base_anos['pedra'] = base_anos['pedra'].replace('Agata', 'Ágata')

In [470]:
base_anos["genero"] = base_anos["genero"].apply(genero_norm)

### Colunas finais do dataframe

In [471]:
#Colunas finais:
base_anos.columns

Index(['ra', 'fase', 'turma', 'nome_anonimizado', 'ano_nasc_original',
       'idade_original', 'genero', 'ano_ingresso', 'instituicao_de_ensino',
       'pedra', 'inde', 'no_av', 'iaa', 'ieg', 'ips', 'ida', 'mat', 'por',
       'ing', 'ipv', 'ian', 'fase_ideal_original', 'defasagem', 'ano_aba',
       'ano_nascimento', 'fase_ideal', 'ipp', 'fase_original', 'idade'],
      dtype='object')

### Salvando o arquivo completo sem nenhuma exclusão de linhas

In [472]:
#base_anos.to_excel('base_anos.xlsx', index=False)
#print("Arquivo 'meus_dados.xlsx' criado com sucesso!")

### Vamos excluir os que não possuem nenhum tipo de informação em Pedra?

In [473]:
#Remoção de linhas onde não temos identificação da informação de PEDRA (optamos por excluir estas linhas para analise das informações)
base_filtrado = base_anos[
    base_anos['pedra'].isna() |
    (base_anos['pedra'].astype(str).str.strip() == '') |
    (base_anos['pedra'].astype(str).str.strip() == 'INCLUIR')
]

In [474]:
#Quantidade de linhas que vamos remover por não ter informações relevantes na coluna PEDRA
base_filtrado.shape

(185, 29)

In [475]:
base_anos_limpo = base_anos.drop(index=base_filtrado.index)

### Salvando o arquivo com exclusão de 185 linhas (coluna PEDRA nula ou em branco ou INCLUIR)

In [476]:
base_anos_limpo.to_excel('base_anos_limpo.xlsx', index=False)
print("Arquivo 'meus_dados.xlsx' criado com sucesso!")

Arquivo 'meus_dados.xlsx' criado com sucesso!


In [477]:
base_anos_limpo

,ra,fase,turma,nome_anonimizado,ano_nasc_original,idade_original,genero,ano_ingresso,instituicao_de_ensino,pedra,...,ipv,ian,fase_ideal_original,defasagem,ano_aba,ano_nascimento,fase_ideal,ipp,fase_original,idade
0,RA-1,7,A,Aluno-1,2003,19,Feminino,2016,Escola Pública,Quartzo,...,7.278,5.0,Fase 8 (Universitários),-1,2022,2003,FASE 8,NaN,NaN,19
1,RA-2,7,A,Aluno-2,2005,17,Feminino,2017,Rede Decisão,Ametista,...,6.778,10.0,Fase 7 (3º EM),0,2022,2005,FASE 7,NaN,NaN,17
2,RA-3,7,A,Aluno-3,2005,17,Feminino,2016,Rede Decisão,Ágata,...,7.556,10.0,Fase 7 (3º EM),0,2022,2005,FASE 7,NaN,NaN,17
3,RA-4,7,A,Aluno-4,2005,17,Masculino,2017,Rede Decisão,Quartzo,...,5.278,10.0,Fase 7 (3º EM),0,2022,2005,FASE 7,NaN,NaN,17
4,RA-5,7,A,Aluno-5,2005,17,Feminino,2016,Rede Decisão,Ametista,...,7.389,10.0,Fase 7 (3º EM),0,2022,2005,FASE 7,NaN,NaN,17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2923,RA-291,FASE 7,7E,Aluno-291,2009-01-17 00:00:00,15,Masculino,2022,Privada - Programa de Apadrinhamento,Ametista,...,7.500,10.0,Fase 5 (1° EM),2,2024,2009,FASE 5,7.5,7E,15
2924,RA-86,FASE 7,7E,Aluno-86,2006-10-04 00:00:00,18,Masculino,2021,Privada *Parcerias com Bolsa 100%,Ametista,...,7.500,10.0,Fase 7 (3° EM),0,2024,2006,FASE 7,7.5,7E,18
2925,RA-143,FASE 7,7E,Aluno-143,2009-08-01 00:00:00,15,Masculino,2021,Privada - Programa de Apadrinhamento,Ametista,...,7.500,10.0,Fase 5 (1° EM),2,2024,2009,FASE 5,7.5,7E,15
2926,RA-166,FASE 7,7E,Aluno-166,2008-06-03 00:00:00,16,Masculino,2021,Privada *Parcerias com Bolsa 100%,Topázio,...,7.500,10.0,Fase 6 (2° EM),1,2024,2008,FASE 6,7.5,7E,16


Após a exportação para arquivos EXCEL realizamos a publicação no GIT para facilitar a utilização de um caminho online dos arquivos

# IMPORTANDO DATAFRAME "LIMPO"

In [478]:
#EXEMPLO PARA IMPORTAR AS BASES EXPORTADAS:
url = "https://raw.githubusercontent.com/vbomura/tc5/main/Codigos/base_anos_limpo.xlsx"
dfBaseAnosLimpos = pd.read_excel(url)